In [6]:
import pandas as pd

# File paths
rxn_path = "/raid/data/smunoz/catnip/data/reaction_table.csv"
protein_path = "/raid/data/smunoz/catnip/ESMC-6000/si_proteins 1.csv"
substrates_path = "/raid/data/smunoz/catnip/data/subsrates.csv"
output_path = "/raid/data/smunoz/catnip/EviCYP/pairwise_reaction_dataset.csv"

# 1. Load Protein Table
df_prot = pd.read_csv(protein_path)
df_prot.columns = df_prot.columns.str.strip()
all_proteins = df_prot["number"].astype(int).unique()

# 2. Load Substrates Table & Map SMILES
df_subs = pd.read_csv(substrates_path)
df_subs.columns = df_subs.columns.str.strip()

# Identify Substrate ID column dynamically
sub_id_col = [
    c for c in df_subs.columns if "substrate" in c.lower() or "id" in c.lower()
][0]

# Filter out missing SMILES or empty structures immediately
subs_mapping = (
    df_subs[[sub_id_col, "SMILES"]]
    .dropna(subset=["SMILES"])
    .drop_duplicates()
)
subs_mapping[sub_id_col] = subs_mapping[sub_id_col].astype(int)

# 3. Load Reaction Table
df_rxn = pd.read_csv(rxn_path)
df_rxn.columns = df_rxn.columns.str.strip()

# Extract positive pairs
positive_pairs = (
    df_rxn[["Substrate = 119", "Enzyme = 163"]]
    .drop_duplicates()
    .dropna()
    .rename(
        columns={"Substrate = 119": "Substrate_ID", "Enzyme = 163": "Enzyme_ID"}
    )
)
positive_pairs["Substrate_ID"] = positive_pairs["Substrate_ID"].astype(int)
positive_pairs["Enzyme_ID"] = positive_pairs["Enzyme_ID"].astype(int)
positive_pairs["Y"] = 1

# Extract unique substrates present in BOTH reaction table AND valid SMILES mapping
rxn_substrates = set(positive_pairs["Substrate_ID"].unique())
valid_smiles_substrates = set(subs_mapping[sub_id_col].unique())

# Substrates kept: Must have a reaction entry AND a valid SMILES
usable_substrates = sorted(
    list(rxn_substrates.intersection(valid_smiles_substrates))
)
dropped_substrates = rxn_substrates - valid_smiles_substrates

if dropped_substrates:
    print(
        f"Warning: Dropped {len(dropped_substrates)} substrate(s) due to missing SMILES: {sorted(list(dropped_substrates))}"
    )

# 4. Generate Cartesian Product (Valid Substrates x Proteins)
full_grid = []
for sub_id in usable_substrates:
    for enz_id in all_proteins:
        full_grid.append({"Substrate_ID": sub_id, "Enzyme_ID": enz_id})

df_grid = pd.DataFrame(full_grid)

# 5. Merge Positive Hits
dataset = pd.merge(
    df_grid, positive_pairs, on=["Substrate_ID", "Enzyme_ID"], how="left"
)
dataset["Y"] = dataset["Y"].fillna(0).astype(int)

# 6. Append SMILES to final dataset
dataset = dataset.merge(
    subs_mapping, left_on="Substrate_ID", right_on=sub_id_col, how="left"
)

# Clean up duplicate key column if names differed
if sub_id_col != "Substrate_ID":
    dataset = dataset.drop(columns=[sub_id_col])

# Reorder columns to standard format: Substrate_ID, Enzyme_ID, SMILES, Y
dataset = dataset[["Substrate_ID", "Enzyme_ID", "SMILES", "Y"]]
dataset = dataset.sort_values(by=["Substrate_ID", "Enzyme_ID"]).reset_index(
    drop=True
)

# Display Summary
print("\n--- DATASET SUMMARY ---")
print(f"Total Substrates Kept: {len(usable_substrates)}")
print(f"Total Unique Proteins: {len(all_proteins)}")
print(f"Total Pairwise Combinations: {len(dataset)}")
print(f"Total Positive Hits (Y = 1): {dataset['Y'].sum()}")
print(f"Total Negative Reactions (Y = 0): {(dataset['Y'] == 0).sum()}")

# Export output CSV
dataset.to_csv(output_path, index=False)
print(f"\nPairwise dataset saved to: {output_path}")


--- DATASET SUMMARY ---
Total Substrates Kept: 118
Total Unique Proteins: 314
Total Pairwise Combinations: 37052
Total Positive Hits (Y = 1): 351
Total Negative Reactions (Y = 0): 36701

Pairwise dataset saved to: /raid/data/smunoz/catnip/EviCYP/pairwise_reaction_dataset.csv


In [7]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Define paths
INPUT_PATH = "/raid/data/smunoz/catnip/EviCYP/pairwise_reaction_dataset.csv"
OUTPUT_DIR = "/raid/data/smunoz/catnip/EviCYP/data_splits"

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load dataset
df = pd.read_csv(INPUT_PATH)

# First split: 80% train, 20% temporary (for val + test)
train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42)

# Second split: Split the 20% temp dataset equally into 10% val and 10% test
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

# Output split sizes to verify ratios
print(f"Total Rows: {len(df)}")
print(
    f"Train Set: {len(train_df)} rows ({len(train_df)/len(df)*100:.1f}%)"
)  # ~80%
print(
    f"Val Set:   {len(val_df)} rows ({len(val_df)/len(df)*100:.1f}%)"
)  # ~10%
print(
    f"Test Set:  {len(test_df)} rows ({len(test_df)/len(df)*100:.1f}%)"
)  # ~10%

# Save files as requested
train_df.to_csv(os.path.join(OUTPUT_DIR, "train.csv"), index=False)
val_df.to_csv(os.path.join(OUTPUT_DIR, "val.csv"), index=False)
test_df.to_csv(os.path.join(OUTPUT_DIR, "test.csv"), index=False)

print(f"\nAll splits successfully saved to: {OUTPUT_DIR}")

Total Rows: 37052
Train Set: 29641 rows (80.0%)
Val Set:   3705 rows (10.0%)
Test Set:  3706 rows (10.0%)

All splits successfully saved to: /raid/data/smunoz/catnip/EviCYP/data_splits


In [9]:
import os
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors

# File paths
TRAIN_PATH = "/raid/data/smunoz/catnip/EviCYP/data_splits/train.csv"
OUTPUT_DIR = "/raid/data/smunoz/catnip/EviCYP/data_splits"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "normalization_parameters.csv")

# 1. Load train dataset directly
train_df = pd.read_csv(TRAIN_PATH)
train_df.columns = train_df.columns.str.strip()

# Extract unique SMILES present in the training set
unique_smiles = train_df["SMILES"].dropna().drop_duplicates().tolist()
print(f"Calculating descriptors for {len(unique_smiles)} unique molecules...")

# 2. Build filtered descriptor list (excluding problematic descriptors)
EXCLUDED_DESCRIPTORS = {
    "SMR_VSA8",
    "SlogP_VSA9",
    "fr_isocyan",
    "fr_prisulfonamd",
    "Ipc",
}
rdkit_desc_list = [
    (name, func)
    for name, func in Descriptors._descList
    if name not in EXCLUDED_DESCRIPTORS
]
descriptor_names = [name for name, _ in rdkit_desc_list]


# Function to compute descriptors for a single molecule
def get_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [np.nan] * len(descriptor_names)
    return [func(mol) for _, func in rdkit_desc_list]


# 3. Calculate descriptor matrix
descriptor_data = [get_descriptors(s) for s in unique_smiles]
desc_df = pd.DataFrame(descriptor_data, columns=descriptor_names)

# Drop any descriptors that contain NaNs across all samples
desc_df = desc_df.dropna(axis=1, how="all")

# 4. Compute mean and standard deviation for each descriptor
stats = []
for col in desc_df.columns:
    mean_val = desc_df[col].mean()
    std_val = desc_df[col].std()

    # Handle zero standard deviation or NaN to prevent division by zero
    if std_val == 0 or np.isnan(std_val):
        std_val = 1.0

    stats.append({"descriptor": col, "mean": mean_val, "std": std_val})

norm_params_df = pd.DataFrame(stats)

# 5. Save to CSV
os.makedirs(OUTPUT_DIR, exist_ok=True)
norm_params_df.to_csv(OUTPUT_PATH, index=False)

print(
    f"Successfully calculated parameters for {len(norm_params_df)} descriptors."
)
print(f"Saved normalization parameters to: {OUTPUT_PATH}")

# Display sample output
print("\nSample Normalization Parameters:")
print(norm_params_df.head(10))

Calculating descriptors for 118 unique molecules...
Successfully calculated parameters for 212 descriptors.
Saved normalization parameters to: /raid/data/smunoz/catnip/EviCYP/data_splits/normalization_parameters.csv

Sample Normalization Parameters:
            descriptor        mean         std
0    MaxAbsEStateIndex   10.766767    2.749540
1       MaxEStateIndex   10.766767    2.749540
2    MinAbsEStateIndex    0.161333    0.242901
3       MinEStateIndex   -0.878099    1.105655
4                  qed    0.537975    0.191461
5                  SPS   29.032196   14.306733
6                MolWt  300.833881  150.752233
7       HeavyAtomMolWt  280.050288  141.674727
8           ExactMolWt  300.605272  150.650182
9  NumValenceElectrons  115.796610   57.399717
